In [2]:
!pip show torch

Name: torch
Version: 2.13.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License-Expression: Apache-2.0 AND Apache-2.0 WITH LLVM-exception AND BSD-2-Clause AND BSD-3-Clause AND BSL-1.0 AND MIT
Location: /Users/admin/anaconda3/envs/qolda/lib/python3.12/site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate, sentence-transformers, torchvision


In [3]:
import re
from collections import Counter

import numpy as np
from gensim.models import Word2Vec

word2vec = Word2Vec.load("models/word2vec.model")

a = "қосымша өте жақсы емес"      # "the app is not very good"
b = "емес жақсы өте қосымша"      # the same four words, shuffled

def bag_of_words(text):
    return Counter(re.findall(r"\w+", text.lower()))

def document_vector(text):
    words = re.findall(r"\w+", text.lower())
    return np.mean([word2vec.wv[w] for w in words if w in word2vec.wv], axis=0)

print("«" + a + "»")
print("«" + b + "»")
print()
print("same bag of words     :", bag_of_words(a) == bag_of_words(b))
print("same TF-IDF row       :", bag_of_words(a) == bag_of_words(b), "(it is built from the counts)")
print("same document vector  :", bool(np.allclose(document_vector(a), document_vector(b))))
print("largest difference    :", float(np.abs(document_vector(a) - document_vector(b)).max()),
      "— floating-point noise, not meaning")
print()
print("Every model in Lectures 2, 3 and 4 sees these two sentences as one input.")

«қосымша өте жақсы емес»
«емес жақсы өте қосымша»

same bag of words     : True
same TF-IDF row       : True (it is built from the counts)
same document vector  : True
largest difference    : 2.9802322387695312e-08 — floating-point noise, not meaning

Every model in Lectures 2, 3 and 4 sees these two sentences as one input.


In [4]:
import os

from dotenv import load_dotenv

load_dotenv(".env")

from datasets import load_dataset

CONFIGS = {"polarity": "polarity_classification", "score": "score_classification"}
NAMES = {"polarity": ["negative", "positive"],
         "score": ["1 star", "2 stars", "3 stars", "4 stars", "5 stars"]}

for task, config in CONFIGS.items():
    df = load_dataset("issai/kazsandra", config, split="train",
                      token=os.getenv("HF_TOKEN")).to_pandas()
    counts = df["label"].value_counts().sort_index()
    print(f"{task}: {len(df)} reviews, {len(counts)} classes")
    for label, n in counts.items():
        print(f"  {NAMES[task][label]:<9} {n:>7}  ({100 * n / len(df):.1f}%)")
    print()

df = load_dataset("issai/kazsandra", CONFIGS["score"], split="train",
                  token=os.getenv("HF_TOKEN")).to_pandas()
print("one review per score:")
for label in sorted(df["label"].unique()):
    text = df[df["label"] == label]["text_cleaned"].iloc[0]
    print(f"  {NAMES['score'][label]:<9} «{str(text)[:52]}»")

/Users/admin/anaconda3/envs/qolda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using the latest cached version of the dataset since issai/kazsandra couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'polarity_classification' at /Users/admin/.cache/huggingface/datasets/issai___kazsandra/polarity_classification/0.0.0/a39e47875861b4f3bdf697b5fb768a1fdd95869a (last modified on Sun Aug 30 21:27:26 2026).


polarity: 134368 reviews, 2 classes
  negative    23951  (17.8%)
  positive   110417  (82.2%)



Using the latest cached version of the dataset since issai/kazsandra couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'score_classification' at /Users/admin/.cache/huggingface/datasets/issai___kazsandra/score_classification/0.0.0/a39e47875861b4f3bdf697b5fb768a1fdd95869a (last modified on Sun Sep  6 12:38:21 2026).


score: 140126 reviews, 5 classes
  1 star      20031  (14.3%)
  2 stars      3920  (2.8%)
  3 stars      5758  (4.1%)
  4 stars      9115  (6.5%)
  5 stars    101302  (72.3%)



Using the latest cached version of the dataset since issai/kazsandra couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'score_classification' at /Users/admin/.cache/huggingface/datasets/issai___kazsandra/score_classification/0.0.0/a39e47875861b4f3bdf697b5fb768a1fdd95869a (last modified on Sun Sep  6 12:38:21 2026).


one review per score:
  1 star    «дұрыс жұмыс жасамайды аймақты басасың басқа жақ шыға»
  2 stars   «карта ашылмайды сондықтан әзірге 2 жұлдыз кейін қолд»
  3 stars   «мен ном жазсам ашпайт не істим»
  4 stars   «құрметті разроботчиктар мына програмада басында тым »
  5 stars   «керемет мотивация мен жаңа көзқарас қалыптастыратын »


In [5]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# A tiny RNN so the numbers fit on the page: 3-dimensional words, 4-dimensional state.
rnn = nn.RNN(input_size=3, hidden_size=4, batch_first=True)

words = torch.tensor([[[1.0, 0.0, 0.0],    # қосымша
                       [0.0, 1.0, 0.0],    # өте
                       [0.0, 0.0, 1.0],    # жақсы
                       [1.0, 1.0, 0.0]]])  # емес

# --- what nn.RNN does, written out ----------------------------------------
W_ih, W_hh = rnn.weight_ih_l0, rnn.weight_hh_l0
b_ih, b_hh = rnn.bias_ih_l0, rnn.bias_hh_l0

h = torch.zeros(4)                                      # the state starts at zero
print("h0 (before any word):", h.numpy().round(3))
for step in range(4):
    x = words[0, step]
    h = torch.tanh(W_ih @ x + b_ih + W_hh @ h + b_hh)   # <- the whole recurrence
    print(f"h{step + 1} after word {step + 1}   :", h.detach().numpy().round(3))

# --- the same thing, done by PyTorch --------------------------------------
outputs, last = rnn(words)
print("\nPyTorch's last hidden state:", last[0, 0].detach().numpy().round(3))
print("matches our loop           :", bool(torch.allclose(h, last[0, 0], atol=1e-6)))
print("\noutputs holds every step   :", tuple(outputs.shape), "= (batch, words, hidden)")
print("its last row is the last hidden state:",
      bool(torch.allclose(outputs[0, -1], last[0, 0])))

h0 (before any word): [0. 0. 0. 0.]
h1 after word 1   : [-0.567 -0.62  -0.504  0.096]
h2 after word 2   : [ 0.208 -0.603 -0.397 -0.419]
h3 after word 3   : [-0.704 -0.136 -0.725 -0.167]
h4 after word 4   : [ 0.152 -0.672 -0.318 -0.251]

PyTorch's last hidden state: [ 0.152 -0.672 -0.318 -0.251]
matches our loop           : True

outputs holds every step   : (1, 4, 4) = (batch, words, hidden)
its last row is the last hidden state: True


In [6]:
LENGTH, DIM, HIDDEN = 40, 32, 64

def gradient_by_position(cell):
    """How much does the final state still depend on the word at each position?"""
    torch.manual_seed(0)
    words = torch.randn(1, LENGTH, DIM, requires_grad=True)
    _, state = cell(words)
    hidden = state[0] if isinstance(state, tuple) else state   # LSTM returns (h, c)
    hidden.sum().backward()                    # one signal, sent back from the end
    return words.grad[0].norm(dim=1)           # one number per position

torch.manual_seed(0)
rnn = nn.RNN(DIM, HIDDEN, batch_first=True)
torch.manual_seed(0)
lstm = nn.LSTM(DIM, HIDDEN, batch_first=True)

# The standard trick: start the forget gate open. nn.LSTM stacks the gates as
# (input, forget, cell, output), so the forget bias is the second quarter.
torch.manual_seed(0)
open_lstm = nn.LSTM(DIM, HIDDEN, batch_first=True)
with torch.no_grad():
    open_lstm.bias_ih_l0[HIDDEN:2 * HIDDEN].fill_(3.0)      # sigmoid(3) ≈ 0.95

grads = {name: gradient_by_position(cell) for name, cell in
         (("RNN", rnn), ("LSTM", lstm), ("LSTM, forget open", open_lstm))}

print(f"{'steps back':>10} {'RNN':>10} {'LSTM':>10} {'LSTM, forget open':>19}")
for back in (0, 1, 2, 5, 10, 20, 30, 39):
    i = LENGTH - 1 - back
    row = "  ".join(f"{g[i]:>8.1e}" for g in grads.values())
    print(f"{back:>10}   {row}")

print("\nhow much of the last word's signal still reaches the first word:")
for name, g in grads.items():
    ratio = float(g[0] / g[-1])
    per_step = ratio ** (1 / (LENGTH - 1))
    print(f"  {name:<18} {ratio:>9.1e}   ×{per_step:.2f} per step")

print("\nThe RNN loses half the signal at every step, so after 39 there is nothing")
print("left. The LSTM's default init is barely better. But hold the forget gate")
print("open and the factor becomes 1.00 — the signal survives the whole sequence.")
print("That knob is the entire point of the cell state, and an LSTM can learn it.")

steps back        RNN       LSTM   LSTM, forget open
         0    3.1e+00   8.0e-01   7.0e-01
         1    1.0e+00   4.0e-01   5.4e-01
         2    5.6e-01   1.8e-01   4.7e-01
         5    6.6e-02   3.6e-02   4.7e-01
        10    1.8e-03   2.6e-03   4.1e-01
        20    9.2e-07   2.1e-05   3.9e-01
        30    6.7e-10   9.5e-08   4.6e-01
        39    1.7e-12   1.5e-09   5.8e-01

how much of the last word's signal still reaches the first word:
  RNN                  5.4e-13   ×0.48 per step
  LSTM                 1.9e-09   ×0.60 per step
  LSTM, forget open    8.3e-01   ×1.00 per step

The RNN loses half the signal at every step, so after 39 there is nothing
left. The LSTM's default init is barely better. But hold the forget gate
open and the factor becomes 1.00 — the signal survives the whole sequence.
That knob is the entire point of the cell state, and an LSTM can learn it.


In [7]:
torch.manual_seed(0)

lstm = nn.LSTM(input_size=3, hidden_size=4, batch_first=True)
words = torch.randn(1, 6, 3)

# nn.LSTM stacks the four gates into one matrix, in this order:
i_w, f_w, g_w, o_w = lstm.weight_ih_l0.chunk(4)
i_u, f_u, g_u, o_u = lstm.weight_hh_l0.chunk(4)
i_b, f_b, g_b, o_b = (lstm.bias_ih_l0 + lstm.bias_hh_l0).chunk(4)

h = torch.zeros(4)
c = torch.zeros(4)
print(f"{'step':>4}  {'forget':>22}  {'input':>22}  {'cell state c':>26}")
for t in range(6):
    x = words[0, t]
    f = torch.sigmoid(f_w @ x + f_u @ h + f_b)      # what to keep from c
    i = torch.sigmoid(i_w @ x + i_u @ h + i_b)      # how much of the new value to add
    g = torch.tanh(g_w @ x + g_u @ h + g_b)         # the candidate value
    o = torch.sigmoid(o_w @ x + o_u @ h + o_b)      # what to reveal
    c = f * c + i * g                               # <- the highway: multiply, then add
    h = o * torch.tanh(c)                           # <- what the next layer sees
    print(f"{t + 1:>4}  {str(f.detach().numpy().round(2)):>22}"
          f"  {str(i.detach().numpy().round(2)):>22}"
          f"  {str(c.detach().numpy().round(2)):>26}")

_, (h_torch, c_torch) = lstm(words)
print("\nmatches nn.LSTM — h:", bool(torch.allclose(h, h_torch[0, 0], atol=1e-6)),
      " c:", bool(torch.allclose(c, c_torch[0, 0], atol=1e-6)))
print("\nforget values near 1 keep a memory alive; near 0 erase it.")
print("c is only ever multiplied by f and added to — never passed through a matrix.")

step                  forget                   input                cell state c
   1   [0.39 0.53 0.63 0.52]   [0.36 0.54 0.57 0.4 ]   [ 0.13  0.02 -0.08  0.16]
   2   [0.35 0.52 0.55 0.43]   [0.5  0.45 0.66 0.4 ]   [ 0.03 -0.09  0.26  0.25]
   3   [0.28 0.49 0.52 0.55]   [0.48 0.4  0.61 0.42]   [ 0.01 -0.25  0.22  0.22]
   4   [0.41 0.49 0.65 0.54]   [0.36 0.55 0.54 0.4 ]   [ 0.11 -0.09  0.04  0.26]
   5   [0.51 0.45 0.7  0.53]   [0.31 0.59 0.45 0.45]   [ 0.12  0.19 -0.2   0.45]
   6   [0.49 0.6  0.76 0.45]   [0.3  0.65 0.55 0.37]   [ 0.24  0.38 -0.25  0.33]

matches nn.LSTM — h: True  c: True

forget values near 1 keep a memory alive; near 0 erase it.
c is only ever multiplied by f and added to — never passed through a matrix.


In [9]:
import pandas as pd

df = load_dataset("issai/kazsandra", "polarity_classification", split="train",
                  token=os.getenv("HF_TOKEN")).to_pandas()
train = pd.concat([g.sample(4000, random_state=42) for _, g in df.groupby("label")])

def tokenize(text):
    return re.findall(r"\w+", str(text).lower())

counts = {}
for text in train["text_cleaned"]:
    for word in tokenize(text):
        counts[word] = counts.get(word, 0) + 1

MIN_COUNT = 2
kept = sorted((w for w, c in counts.items() if c >= MIN_COUNT),
              key=lambda w: (-counts[w], w))

vocab = {"<pad>": 0, "<unk>": 1}          # two reserved slots
for word in kept:
    vocab[word] = len(vocab)

print("distinct words in the training text:", len(counts))
print("kept (seen at least twice)         :", len(kept))
print("vocabulary with <pad> and <unk>    :", len(vocab))
print()
print("the ten most frequent words:")
for word in kept[:10]:
    print(f"  {vocab[word]:>3}  {word:<10} {counts[word]}")
print()
print("«жақсы» is index", vocab["жақсы"], "— a word not in the table becomes", vocab["<unk>"])

Using the latest cached version of the dataset since issai/kazsandra couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'polarity_classification' at /Users/admin/.cache/huggingface/datasets/issai___kazsandra/polarity_classification/0.0.0/a39e47875861b4f3bdf697b5fb768a1fdd95869a (last modified on Sun Aug 30 21:27:26 2026).


distinct words in the training text: 19201
kept (seen at least twice)         : 5914
vocabulary with <pad> and <unk>    : 5916

the ten most frequent words:
    2  өте        1680
    3  керемет    1156
    4  жақсы      986
    5  маған      839
    6  екен       776
    7  ұнады      755
    8  мен        726
    9  ойын       715
   10  бұл        505
   11  деп        472

«жақсы» is index 4 — a word not in the table becomes 1


In [10]:
PAD = 0
torch.manual_seed(0)

embedding = nn.Embedding(5916, 100, padding_idx=PAD)
lstm = nn.LSTM(100, 128, batch_first=True)
out = nn.Linear(128, 2)

ids = torch.tensor([[142, 7, 39, 0, 0],       # 3 real words, 2 padding
                    [12, 88, 4, 501, 63],     # 5 real words
                    [88, 0, 0, 0, 0]])        # 1 real word
lengths = torch.tensor([3, 5, 1])

print("ids                          ", tuple(ids.shape), "  integers")
e = embedding(ids)
print("after Embedding              ", tuple(e.shape), " one 100-number vector per word")
packed = nn.utils.rnn.pack_padded_sequence(e, lengths, batch_first=True, enforce_sorted=False)
print("after pack_padded_sequence   ", tuple(packed.data.shape), "  only the 9 real words")
_, (h, c) = lstm(packed)
print("hidden state h               ", tuple(h.shape), "  (layers, batch, hidden)")
print("cell state c                 ", tuple(c.shape), "  same shape, stays inside")
last = h[-1]
print("h[-1] — the last layer       ", tuple(last.shape), "  one vector per review")
logits = out(last)
print("after Linear                 ", tuple(logits.shape), "    one score per class")
print("\nlogits:\n", logits.detach().numpy().round(3))
print("\npredicted class per review:", logits.argmax(1).tolist())
print("\n5 words in, 2 numbers out. Everything the model decided about a review")
print("has to fit through the 128 numbers of h[-1].")

ids                           (3, 5)   integers
after Embedding               (3, 5, 100)  one 100-number vector per word
after pack_padded_sequence    (9, 100)   only the 9 real words
hidden state h                (1, 3, 128)   (layers, batch, hidden)
cell state c                  (1, 3, 128)   same shape, stays inside
h[-1] — the last layer        (3, 128)   one vector per review
after Linear                  (3, 2)     one score per class

logits:
 [[ 0.016 -0.012]
 [ 0.018 -0.15 ]
 [-0.037 -0.159]]

predicted class per review: [0, 0, 0]

5 words in, 2 numbers out. Everything the model decided about a review
has to fit through the 128 numbers of h[-1].


In [11]:
PAD = 0

class Recurrent(nn.Module):
    """Embedding -> RNN or LSTM -> Linear."""

    def __init__(self, vocab_size, classes, kind="lstm", dim=100, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=PAD)
        cell = nn.LSTM if kind == "lstm" else nn.RNN
        self.rnn = cell(dim, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, classes)

    def forward(self, ids, lengths):
        embedded = self.dropout(self.embedding(ids))          # (batch, words, dim)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths, batch_first=True, enforce_sorted=False)
        _, state = self.rnn(packed)
        hidden = state[0] if isinstance(state, tuple) else state   # LSTM returns (h, c)
        return self.out(self.dropout(hidden[-1]))             # (batch, classes)


torch.manual_seed(42)
model = Recurrent(vocab_size=5916, classes=2, kind="lstm")   # the vocabulary above
print(model)
print()
for name, p in model.named_parameters():
    print(f"  {name:<22} {tuple(p.shape)!s:<14} {p.numel():>9,}")
print(f"\ntotal: {sum(p.numel() for p in model.parameters()):,} parameters")

ids = torch.randint(2, 5916, (4, 7))
lengths = torch.tensor([7, 5, 3, 1])
print("\nforward pass:", tuple(ids.shape), "->", tuple(model(ids, lengths).shape))

Recurrent(
  (embedding): Embedding(5916, 100, padding_idx=0)
  (rnn): LSTM(100, 128, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (out): Linear(in_features=128, out_features=2, bias=True)
)

  embedding.weight       (5916, 100)      591,600
  rnn.weight_ih_l0       (512, 100)        51,200
  rnn.weight_hh_l0       (512, 128)        65,536
  rnn.bias_ih_l0         (512,)               512
  rnn.bias_hh_l0         (512,)               512
  out.weight             (2, 128)             256
  out.bias               (2,)                   2

total: 709,618 parameters

forward pass: (4, 7) -> (4, 2)


In [12]:
import os
import re
import time

from dotenv import load_dotenv

load_dotenv(".env")

import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader

PAD, UNK, SEED = 0, 1, 42

# ---- data ----------------------------------------------------------------
def split(name, per_class):
    df = load_dataset("issai/kazsandra", "polarity_classification", split=name,
                      token=os.getenv("HF_TOKEN")).to_pandas()
    df = pd.concat([g.sample(per_class, random_state=SEED) for _, g in df.groupby("label")])
    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)

def tokenize(text):
    return re.findall(r"\w+", str(text).lower())

train_df, test_df = split("train", 4000), split("test", 1000)

counts = {}
for text in train_df["text_cleaned"]:
    for word in tokenize(text):
        counts[word] = counts.get(word, 0) + 1
vocab = {"<pad>": PAD, "<unk>": UNK}
for word in sorted((w for w, c in counts.items() if c >= 2), key=lambda w: (-counts[w], w)):
    vocab[word] = len(vocab)

def encode(df, max_len=60):
    rows = []
    for text, label in zip(df["text_cleaned"], df["label"]):
        ids = [vocab.get(w, UNK) for w in tokenize(text)][:max_len]
        rows.append((ids or [UNK], int(label)))
    return rows

def collate(batch):
    lengths = torch.tensor([len(ids) for ids, _ in batch])
    padded = torch.full((len(batch), int(lengths.max())), PAD, dtype=torch.long)
    for i, (ids, _) in enumerate(batch):
        padded[i, :len(ids)] = torch.tensor(ids)
    return padded, lengths, torch.tensor([y for _, y in batch])

train_rows, test_rows = encode(train_df), encode(test_df)
test_batches = DataLoader(test_rows, 64, shuffle=False, collate_fn=collate)

def shuffled_batches():
    """A fresh generator each time, so every model sees the same batch order."""
    return DataLoader(train_rows, 64, shuffle=True, collate_fn=collate,
                      generator=torch.Generator().manual_seed(SEED))

# ---- model ---------------------------------------------------------------
class Recurrent(nn.Module):
    def __init__(self, vocab_size, classes, kind, dim=100, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=PAD)
        self.rnn = (nn.LSTM if kind == "lstm" else nn.RNN)(dim, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, classes)

    def forward(self, ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            self.dropout(self.embedding(ids)), lengths, batch_first=True, enforce_sorted=False)
        _, state = self.rnn(packed)
        hidden = state[0] if isinstance(state, tuple) else state
        return self.out(self.dropout(hidden[-1]))

# ---- train ---------------------------------------------------------------
def accuracy(model, batches):
    model.eval()
    right = total = 0
    with torch.no_grad():
        for ids, lengths, labels in batches:
            right += int((model(ids, lengths).argmax(1) == labels).sum())
            total += len(labels)
    return right / total

print(f"train {len(train_df)}  test {len(test_df)}  vocabulary {len(vocab)}\n")

for kind in ("rnn", "lstm"):
    torch.manual_seed(SEED)
    model = Recurrent(len(vocab), 2, kind)
    loss_fn, optimiser = nn.CrossEntropyLoss(), torch.optim.Adam(model.parameters(), lr=1e-3)
    print(f"{kind.upper()}  {sum(p.numel() for p in model.parameters()):,} parameters")
    train_batches = shuffled_batches()
    for epoch in range(1, 6):
        model.train()
        started, running = time.time(), 0.0
        for ids, lengths, labels in train_batches:
            optimiser.zero_grad()
            loss = loss_fn(model(ids, lengths), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimiser.step()
            running += loss.item() * len(labels)
        print(f"  epoch {epoch}  loss={running / len(train_df):.4f}"
              f"  test={accuracy(model, test_batches):.3f}  ({time.time() - started:.0f}s)")
    print()

Using the latest cached version of the dataset since issai/kazsandra couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'polarity_classification' at /Users/admin/.cache/huggingface/datasets/issai___kazsandra/polarity_classification/0.0.0/a39e47875861b4f3bdf697b5fb768a1fdd95869a (last modified on Sun Aug 30 21:27:26 2026).
Using the latest cached version of the dataset since issai/kazsandra couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'polarity_classification' at /Users/admin/.cache/huggingface/datasets/issai___kazsandra/polarity_classification/0.0.0/a39e47875861b4f3bdf697b5fb768a1fdd95869a (last modified on Sun Aug 30 21:27:26 2026).


train 8000  test 2000  vocabulary 5916

RNN  621,298 parameters
  epoch 1  loss=0.6460  test=0.711  (1s)
  epoch 2  loss=0.5634  test=0.728  (1s)
  epoch 3  loss=0.5335  test=0.696  (1s)
  epoch 4  loss=0.5112  test=0.751  (1s)
  epoch 5  loss=0.4823  test=0.755  (1s)

LSTM  709,618 parameters
  epoch 1  loss=0.6083  test=0.727  (3s)
  epoch 2  loss=0.5121  test=0.756  (3s)
  epoch 3  loss=0.4741  test=0.756  (3s)
  epoch 4  loss=0.4369  test=0.770  (3s)
  epoch 5  loss=0.4099  test=0.772  (2s)



In [13]:
import os
import re
import time

from dotenv import load_dotenv

load_dotenv(".env")

import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from gensim.models import Word2Vec
from torch.utils.data import DataLoader

PAD, UNK, SEED = 0, 1, 42

# ---- data ----------------------------------------------------------------
def split(name, per_class):
    df = load_dataset("issai/kazsandra", "polarity_classification", split=name,
                      token=os.getenv("HF_TOKEN")).to_pandas()
    df = pd.concat([g.sample(per_class, random_state=SEED) for _, g in df.groupby("label")])
    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)

def tokenize(text):
    return re.findall(r"\w+", str(text).lower())

train_df, test_df = split("train", 4000), split("test", 1000)

counts = {}
for text in train_df["text_cleaned"]:
    for word in tokenize(text):
        counts[word] = counts.get(word, 0) + 1
vocab = {"<pad>": PAD, "<unk>": UNK}
for word in sorted((w for w, c in counts.items() if c >= 2), key=lambda w: (-counts[w], w)):
    vocab[word] = len(vocab)

def encode(df, max_len=60):
    rows = []
    for text, label in zip(df["text_cleaned"], df["label"]):
        ids = [vocab.get(w, UNK) for w in tokenize(text)][:max_len]
        rows.append((ids or [UNK], int(label)))
    return rows

def collate(batch):
    lengths = torch.tensor([len(ids) for ids, _ in batch])
    padded = torch.full((len(batch), int(lengths.max())), PAD, dtype=torch.long)
    for i, (ids, _) in enumerate(batch):
        padded[i, :len(ids)] = torch.tensor(ids)
    return padded, lengths, torch.tensor([y for _, y in batch])

train_rows, test_rows = encode(train_df), encode(test_df)
test_batches = DataLoader(test_rows, 64, shuffle=False, collate_fn=collate)

def shuffled_batches():
    """A fresh generator each time, so every model sees the same batch order."""
    return DataLoader(train_rows, 64, shuffle=True, collate_fn=collate,
                      generator=torch.Generator().manual_seed(SEED))

# ---- model ---------------------------------------------------------------
class Recurrent(nn.Module):
    def __init__(self, vocab_size, classes, kind, dim=100, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=PAD)
        self.rnn = (nn.LSTM if kind == "lstm" else nn.RNN)(dim, hidden, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(hidden, classes)

    def forward(self, ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            self.dropout(self.embedding(ids)), lengths, batch_first=True, enforce_sorted=False)
        _, state = self.rnn(packed)
        hidden = state[0] if isinstance(state, tuple) else state
        return self.out(self.dropout(hidden[-1]))

# ---- train ---------------------------------------------------------------
def accuracy(model, batches):
    model.eval()
    right = total = 0
    with torch.no_grad():
        for ids, lengths, labels in batches:
            right += int((model(ids, lengths).argmax(1) == labels).sum())
            total += len(labels)
    return right / total

vectors = Word2Vec.load("models/word2vec.model").wv     # trained in Lecture 4

print(f"train {len(train_df)}  test {len(test_df)}  vocabulary {len(vocab)}\n")

for pretrained in (False, True):
    torch.manual_seed(SEED)
    model = Recurrent(len(vocab), 2, "lstm")
    if pretrained:
        found = 0
        for word, index in vocab.items():
            if word in vectors:
                model.embedding.weight.data[index] = torch.tensor(vectors[word])
                found += 1
        print(f"word2vec start   {found} of {len(vocab)} rows filled from Lecture 4")
    else:
        print("random start     every row is noise at epoch 0")
    loss_fn, optimiser = nn.CrossEntropyLoss(), torch.optim.Adam(model.parameters(), lr=1e-3)
    train_batches = shuffled_batches()
    for epoch in range(1, 6):
        model.train()
        started, running = time.time(), 0.0
        for ids, lengths, labels in train_batches:
            optimiser.zero_grad()
            loss = loss_fn(model(ids, lengths), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimiser.step()
            running += loss.item() * len(labels)
        print(f"  epoch {epoch}  loss={running / len(train_df):.4f}"
              f"  test={accuracy(model, test_batches):.3f}  ({time.time() - started:.0f}s)")
    print()

Using the latest cached version of the dataset since issai/kazsandra couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'polarity_classification' at /Users/admin/.cache/huggingface/datasets/issai___kazsandra/polarity_classification/0.0.0/a39e47875861b4f3bdf697b5fb768a1fdd95869a (last modified on Sun Aug 30 21:27:26 2026).
Using the latest cached version of the dataset since issai/kazsandra couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'polarity_classification' at /Users/admin/.cache/huggingface/datasets/issai___kazsandra/polarity_classification/0.0.0/a39e47875861b4f3bdf697b5fb768a1fdd95869a (last modified on Sun Aug 30 21:27:26 2026).


train 8000  test 2000  vocabulary 5916

random start     every row is noise at epoch 0
  epoch 1  loss=0.6083  test=0.727  (2s)
  epoch 2  loss=0.5121  test=0.756  (3s)
  epoch 3  loss=0.4741  test=0.756  (2s)
  epoch 4  loss=0.4369  test=0.770  (3s)
  epoch 5  loss=0.4099  test=0.772  (2s)

word2vec start   5743 of 5916 rows filled from Lecture 4
  epoch 1  loss=0.5083  test=0.785  (3s)
  epoch 2  loss=0.4373  test=0.799  (3s)
  epoch 3  loss=0.4045  test=0.787  (2s)
  epoch 4  loss=0.3808  test=0.794  (3s)
  epoch 5  loss=0.3542  test=0.787  (3s)

